# ESIS Mobile — Kaggle APK Build

**Instructions:**
1. Upload the `mobile/` directory as a Kaggle Dataset named `esis-mobile`
2. Add it as an input to this notebook
3. Run All — build takes ~8–12 minutes
4. Download `esis-debug.apk` from the Output panel

Produces a debug APK installable on any Android phone.

In [ ]:
%%bash
# Cell 1 — Install Node 18 and Java 17
curl -fsSL https://deb.nodesource.com/setup_18.x | sudo -E bash -
sudo apt-get install -y nodejs

node --version
npm --version

sudo apt-get install -y openjdk-17-jdk
java -version

In [ ]:
%%bash
# Cell 2 — Install Android SDK command-line tools
cd /tmp

wget -q https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip \
  -O cmdtools.zip
unzip -q cmdtools.zip -d android-sdk
mkdir -p android-sdk/cmdline-tools/latest
mv android-sdk/cmdline-tools/cmdline-tools/* android-sdk/cmdline-tools/latest/
export ANDROID_HOME=/tmp/android-sdk
export PATH=$PATH:$ANDROID_HOME/cmdline-tools/latest/bin:$ANDROID_HOME/platform-tools

yes | sdkmanager --licenses 2>/dev/null | tail -1
sdkmanager "platform-tools" "build-tools;34.0.0" "platforms;android-34" 2>&1 | tail -5
echo "Android SDK installed."

In [ ]:
%%bash
# Cell 3 — Copy mobile/ source and install npm deps
# Kaggle dataset input is mounted at /kaggle/input/esis-mobile/
cp -r /kaggle/input/esis-mobile/mobile /tmp/esis-mobile
cd /tmp/esis-mobile
npm install --legacy-peer-deps 2>&1 | tail -5
echo "npm install done"

In [ ]:
%%bash
# Cell 4 — Install Expo CLI and prebuild the Android project
export ANDROID_HOME=/tmp/android-sdk
export PATH=$PATH:$ANDROID_HOME/cmdline-tools/latest/bin:$ANDROID_HOME/platform-tools

cd /tmp/esis-mobile

npm install -g expo-cli @expo/ngrok 2>&1 | tail -3

npx expo prebuild --platform android --clean 2>&1 | tail -10
echo "Prebuild complete"

In [ ]:
%%bash
# Cell 5 — Build debug APK with Gradle
export ANDROID_HOME=/tmp/android-sdk
export PATH=$PATH:$ANDROID_HOME/cmdline-tools/latest/bin:$ANDROID_HOME/platform-tools
export JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64

cd /tmp/esis-mobile/android

chmod +x gradlew
./gradlew assembleDebug --no-daemon 2>&1 | tail -20
echo "Build complete"

In [ ]:
import shutil, os

src = '/tmp/esis-mobile/android/app/build/outputs/apk/debug/app-debug.apk'
dst = '/kaggle/working/esis-debug.apk'

shutil.copy(src, dst)
size_mb = os.path.getsize(dst) / 1024 / 1024
print(f'APK ready: {dst} ({size_mb:.1f} MB)')
print('Download from the Kaggle output panel on the right →')